In [1]:
import pandas as pd
import sqlite3

In [2]:
# create database connection
conn = sqlite3.connect('vendor_performance.db')

In [3]:
# checking for table
tables = pd.read_sql_query("SELECT name FROM sqlite_master WHERE type= 'table' ", conn)
tables

,name
0,begin_inventory
1,end_inventory
2,purchases
3,purchase_prices
4,vendor_invoice


In [4]:
for table in tables['name']:
    print('-' * 50, f'{table}', '-' * 50)

    query = f"SELECT COUNT(*) AS count FROM {table}"

    result = pd.read_sql(query, conn)

    print("Count of records:", result['count'].values[0])

-------------------------------------------------- begin_inventory --------------------------------------------------
Count of records: 206529
-------------------------------------------------- end_inventory --------------------------------------------------
Count of records: 223205
-------------------------------------------------- purchases --------------------------------------------------
Count of records: 2372471
-------------------------------------------------- purchase_prices --------------------------------------------------
Count of records: 12260
-------------------------------------------------- vendor_invoice --------------------------------------------------
Count of records: 5543


In [ ]:
# MySQL Question to answer
- How many unique vendors are there?
- What is the average purchase price?
- Which product classifications contribute the most to purchase spending?
- Which vendors have the longest average delivery duration?
- Which products have the highest estimated gross margin percentage?
- Which products have high closing inventory but relatively low purchase activity?
- How does inventory quantity change between opening and closing snapshots for matching inventory IDs?
- Identify supplier price differences
- Compare delivery durations between suppliers


In [9]:
# How many unique vendors are there?
pd.read_sql_query("""SELECT COUNT(DISTINCT VendorNumber) AS Total_Vendors
FROM purchases""",conn)

,Total_Vendors
0,126


In [10]:
# What is the average purchase price?

pd.read_sql_query(
    """SELECT ROUND(AVG(PurchasePrice), 2) AS Average_Purchase_Price
FROM purchases""", conn
)

,Average_Purchase_Price
0,12.05


In [11]:
# Which product classifications contribute the most to purchase spending?
pd.read_sql_query("""SELECT
    Classification,
    ROUND(SUM(Dollars), 2) AS Total_Purchase_Spending
FROM purchases
GROUP BY Classification
ORDER BY Total_Purchase_Spending DESC""", conn)

,Classification,Total_Purchase_Spending
0,1,2.169647e+08
1,2,1.049358e+08


In [15]:
# Which vendors have the longest average delivery duration?
pd.read_sql_query(
    """SELECT
    VendorName,
    ROUND(AVG(JULIANDAY(ReceivingDate, PODate)), 2) AS Avg_Delivery_Days,
    COUNT(DISTINCT PONumber) AS Total_Orders
FROM purchases
WHERE PODate IS NOT NULL
  AND ReceivingDate IS NOT NULL
GROUP BY VendorName
ORDER BY Avg_Delivery_Days DESC
""", conn
)

,VendorName,Avg_Delivery_Days,Total_Orders
0,ZORVINO VINEYARDS,None,55
1,WINE GROUP INC,None,55
2,WILLIAM GRANT & SONS INC,None,55
3,WESTERN SPIRITS BEVERAGE CO,None,55
4,WEIN BAUER INC,None,53
...,...,...,...
124,AMERICAN SPIRITS EXCHANGE,None,6
125,ALTAMAR BRANDS LLC,None,40
126,ALISA CARR BEVERAGES,None,19
127,ADAMBA IMPORTS INTL INC,None,35


In [17]:
# Which products have the highest estimated gross margin percentage?
query = """
SELECT
    Brand,
    Description,
    Size,
    Price AS Selling_Price,
    PurchasePrice AS Purchase_Price,
    ROUND(
        ((Price - PurchasePrice) * 100.0 / Price), 2
    ) AS Gross_Margin_Percentage
FROM purchase_prices
WHERE Price > 0
  AND PurchasePrice IS NOT NULL
ORDER BY Gross_Margin_Percentage DESC
LIMIT 15;
"""

df = pd.read_sql_query(query, conn)
df

,Brand,Description,Size,Selling_Price,Purchase_Price,Gross_Margin_Percentage
0,22975,L.A. Cetto Nebbiolo Priv Rsv,750mL,23.99,3.62,84.91
1,19835,Luiano Rosato,375mL,12.99,2.16,83.37
2,16935,Martinotti Metodo Brut,750mL,17.99,3.10,82.77
3,38467,Bertiol Prosecco Conegliano,750mL,16.99,2.93,82.75
4,18627,Viberti Chard,750mL,16.99,3.01,82.28
5,21884,St Julien Rose CDR,750mL,11.99,2.36,80.32
6,18630,Viberti Langhe Dolba,750mL,13.99,2.97,78.77
7,21616,Moonstone Plum Sake,300mL,5.99,1.31,78.13
8,21855,Waterstone Merlot,750mL,17.99,3.99,77.82
9,19952,Yellow Tail Winter Red,1500mL,14.99,3.35,77.65


In [20]:
# Which products have high closing inventory but relatively low purchase activity?
query = """
WITH closing_stock AS (
    SELECT
        Brand,
        Description,
        Size,
        SUM(onHand) AS Closing_Inventory
    FROM end_inventory
    GROUP BY Brand, Description, Size
),

purchase_activity AS (
    SELECT
        Brand,
        Description,
        Size,
        SUM(Quantity) AS Total_Purchased
    FROM purchases
    GROUP BY Brand, Description, Size
)

SELECT
    c.Brand,
    c.Description,
    c.Size,
    c.Closing_Inventory,
    COALESCE(p.Total_Purchased, 0) AS Total_Purchased
FROM closing_stock c
LEFT JOIN purchase_activity p
    ON c.Brand = p.Brand
    AND c.Description = p.Description
    AND c.Size = p.Size
WHERE c.Closing_Inventory > 0
ORDER BY
    c.Closing_Inventory DESC,
    Total_Purchased ASC
LIMIT 10;
"""

df = pd.read_sql_query(query, conn)
df



,Brand,Description,Size,Closing_Inventory,Total_Purchased
0,3545,Ketel One Vodka,1.75L,16770,138109
1,4261,Capt Morgan Spiced Rum,1.75L,16769,201682
2,8111,Smirnoff 80 Proof,50mL,16242,337660
3,8068,Absolut 80 Proof,1.75L,15608,187407
4,1233,Jack Daniels No 7 Black,1.75L,15047,145080
5,3102,Smirnoff Traveler,1.75L,14208,161386
6,5111,Dr McGillicuddy's Mentholmnt,50mL,13804,244654
7,1376,Jim Beam,1.75L,13180,108866
8,6570,Kendall Jackson Chard Vt RSV,750mL,13016,215668
9,1892,Yukon Jack,50mL,12764,254824


In [22]:
#  How does inventory quantity change between opening and closing snapshots for matching inventory IDs?
query = """
SELECT
    b.InventoryId,
    b.Store,
    b.City,
    b.Brand,
    b.Description,
    b.Size,
    b.onHand AS Opening_Inventory,
    e.onHand AS Closing_Inventory,

    (e.onHand - b.onHand) AS Inventory_Change,

    ROUND(
        (e.onHand - b.onHand) * 100.0 /
        NULLIF(b.onHand, 0), 2
    ) AS Change_Percentage

FROM begin_inventory b
INNER JOIN end_inventory e
    ON b.InventoryId = e.InventoryId

ORDER BY Inventory_Change DESC;
"""

df = pd.read_sql_query(query, conn)
df.head(10)

,InventoryId,Store,City,Brand,Description,Size,Opening_Inventory,Closing_Inventory,Inventory_Change,Change_Percentage
0,67_EANVERNESS_5111,67,EANVERNESS,5111,Dr McGillicuddy's Mentholmnt,50mL,164,3005,2841,1732.32
1,50_MOUNTMEND_3102,50,MOUNTMEND,3102,Smirnoff Traveler,1.75L,300,1227,927,309.00
2,22_SHARNWICK_1892,22,SHARNWICK,1892,Yukon Jack,50mL,140,1038,898,641.43
3,27_MOUNTMEND_126,27,MOUNTMEND,126,Grey Goose Vodka,Liter,15,884,869,5793.33
4,74_PAENTMARWY_8184,74,PAENTMARWY,8184,Kahlua,50mL,41,872,831,2026.83
5,50_MOUNTMEND_3405,50,MOUNTMEND,3405,Tito's Handmade Vodka,1.75L,339,1164,825,243.36
6,74_PAENTMARWY_5708,74,PAENTMARWY,5708,Dr McGillicuddy's Root Beer,50mL,0,715,715,NaN
7,50_MOUNTMEND_1233,50,MOUNTMEND,1233,Jack Daniels No 7 Black,1.75L,537,1246,709,132.03
8,15_WANBORNE_126,15,WANBORNE,126,Grey Goose Vodka,Liter,104,787,683,656.73
9,50_MOUNTMEND_8068,50,MOUNTMEND,8068,Absolut 80 Proof,1.75L,601,1273,672,111.81


In [24]:
# Identify supplier price differences
query = """
SELECT
    p.Brand,
    p.Description,
    p.Size,
    p.VendorName,
    ROUND(AVG(p.PurchasePrice), 2) AS Avg_Purchase_Price,
    COUNT(*) AS Purchase_Records
FROM purchases p
INNER JOIN purchase_prices pp
    ON p.Brand = pp.Brand
    AND p.Description = pp.Description
    AND p.Size = pp.Size
    AND p.VendorNumber = pp.VendorNumber
WHERE p.PurchasePrice > 0
GROUP BY
    p.Brand,
    p.Description,
    p.Size,
    p.VendorName
ORDER BY
    p.Brand,
    p.Description,
    p.Size;
"""

df = pd.read_sql_query(query, conn)
df.head(10)

,Brand,Description,Size,VendorName,Avg_Purchase_Price,Purchase_Records
0,58,Gekkeikan Black & Gold Sake,750mL,SHAW ROSS INT L IMP LTD,9.28,596
1,60,Canadian Club 1858 VAP,750mL,JIM BEAM BRANDS COMPANY,7.40,97
2,61,Margaritaville Silver,750mL,SAZERAC CO INC,10.60,26
3,62,Herradura Silver Tequila,750mL,BROWN-FORMAN CORP,28.67,567
4,63,Herradura Reposado Tequila,750mL,BROWN-FORMAN CORP,30.46,524
5,70,Luxardo Amaretto di Saschira,750mL,ULTRA BEVERAGE COMPANY LLP,13.96,3
6,72,No. 3 London Dry Gin,750mL,ULTRA BEVERAGE COMPANY LLP,26.11,77
7,75,Three Olives Tomato Vodka,750mL,PROXIMO SPIRITS INC.,10.94,1
8,77,Three Olives Espresso Vodka,750mL,PROXIMO SPIRITS INC.,10.39,842
9,79,Three Olives Loopy Vodka,750mL,PROXIMO SPIRITS INC.,9.62,496


In [25]:
# Compare delivery durations between suppliers
query = """
SELECT
    VendorNumber,
    VendorName,
    ROUND(
        AVG(JULIANDAY(ReceivingDate) - JULIANDAY(PODate)), 2
    ) AS Avg_Delivery_Days,
    COUNT(DISTINCT PONumber) AS Total_Orders,
    MIN(JULIANDAY(ReceivingDate) - JULIANDAY(PODate))
        AS Min_Delivery_Days,
    MAX(JULIANDAY(ReceivingDate) - JULIANDAY(PODate))
        AS Max_Delivery_Days
FROM purchases
WHERE PODate IS NOT NULL
  AND ReceivingDate IS NOT NULL
GROUP BY VendorNumber, VendorName
HAVING COUNT(DISTINCT PONumber) >= 10
ORDER BY Avg_Delivery_Days ASC;
"""

df = pd.read_sql_query(query, conn)
df.head(10)

,VendorNumber,VendorName,Avg_Delivery_Days,Total_Orders,Min_Delivery_Days,Max_Delivery_Days
0,4425,MARTIGNETTI COMPANIES,6.70,27,3.0,12.0
1,1703,ALISA CARR BEVERAGES,6.81,19,3.0,12.0
2,1650,Circa Wines,6.95,43,3.0,12.0
3,9206,PHILLIPS PRODUCTS CO.,7.01,55,3.0,13.0
4,8150,SEA HAGG DISTILLERY LLC,7.02,54,3.0,13.0
5,6355,NICHE W & S,7.07,55,3.0,13.0
6,6215,MOONLIGHT MEADERY,7.08,54,3.0,14.0
7,98450,Serralles Usa LLC,7.12,54,3.0,14.0
8,8892,TRINCHERO FAMILY ESTATES,7.13,55,3.0,14.0
9,9622,WEIN BAUER INC,7.13,53,3.0,14.0
